In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import MetaTrader5 as mt5

# Suppress all warnings
warnings.filterwarnings("ignore")

sns.set_style("darkgrid")

## Importing data from MetaTrader5

In [2]:
if not mt5.initialize(r"c:\Users\Omega Joctan\AppData\Roaming\Pepperstone MetaTrader 5\terminal64.exe"):
    print("Failed to initialize MetaTrader5, error =", mt5.last_error())
    quit()

In [3]:
symbol = "EURUSD"
timeframe = mt5.TIMEFRAME_D1

if not mt5.symbol_select(symbol, True):
    print("Failed to select and add a symbol to the MarketWatch, Error = ",mt5.last_error())
    quit()
    
rates = mt5.copy_rates_from_pos(symbol, timeframe, 1, 10000)
df = pd.DataFrame(rates) # convert rates into a pandas dataframe

df

,time,open,high,low,close,tick_volume,spread,real_volume
0,611280000,1.00780,1.01050,1.00630,1.00760,821,50,0
1,611366400,0.99620,1.00580,0.99100,0.99600,2941,50,0
2,611452800,0.99180,0.99440,0.98760,0.99190,1351,50,0
3,611539200,0.99330,0.99370,0.99310,0.99310,101,50,0
4,611798400,0.97360,0.97360,0.97320,0.97360,81,50,0
...,...,...,...,...,...,...,...,...
9995,1748390400,1.13239,1.13453,1.12838,1.12910,153191,0,0
9996,1748476800,1.12918,1.13849,1.12105,1.13659,191948,0,0
9997,1748563200,1.13630,1.13901,1.13127,1.13470,186924,0,0
9998,1748822400,1.13435,1.14500,1.13412,1.14436,168697,0,0


## Creating stationary features

Filter all columns but OHLC

In [4]:
ohlc_df = df.drop(columns=[
    "time",
    "tick_volume",
    "spread",
    "real_volume"
])

ohlc_df

,open,high,low,close
0,1.00780,1.01050,1.00630,1.00760
1,0.99620,1.00580,0.99100,0.99600
2,0.99180,0.99440,0.98760,0.99190
3,0.99330,0.99370,0.99310,0.99310
4,0.97360,0.97360,0.97320,0.97360
...,...,...,...,...
9995,1.13239,1.13453,1.12838,1.12910
9996,1.12918,1.13849,1.12105,1.13659
9997,1.13630,1.13901,1.13127,1.13470
9998,1.13435,1.14500,1.13412,1.14436


In [5]:
stationary_df = pd.DataFrame()

for col in ohlc_df.columns:
    stationary_df["diff_"+col] = ohlc_df[col].diff()

stationary_df.dropna(inplace=True)
stationary_df

,diff_open,diff_high,diff_low,diff_close
1,-0.01160,-0.00470,-0.01530,-0.01160
2,-0.00440,-0.01140,-0.00340,-0.00410
3,0.00150,-0.00070,0.00550,0.00120
4,-0.01970,-0.02010,-0.01990,-0.01950
5,0.00070,0.00170,0.00040,0.00090
...,...,...,...,...
9995,-0.00570,-0.00620,-0.00395,-0.00370
9996,-0.00321,0.00396,-0.00733,0.00749
9997,0.00712,0.00052,0.01022,-0.00189
9998,-0.00195,0.00599,0.00285,0.00966


## Check for stationarity

In [6]:
from statsmodels.tsa.stattools import adfuller

for col in stationary_df.columns:
    
    result = adfuller(stationary_df[col])
    print(f'{col} p-value: {result[1]}')

diff_open p-value: 0.0
diff_high p-value: 8.363342845441995e-29
diff_low p-value: 1.2175705509142461e-28
diff_close p-value: 0.0


Checking for correlation

In [7]:
stationary_df.corr()

,diff_open,diff_high,diff_low,diff_close
diff_open,1.000000,0.567907,0.570190,0.019019
diff_high,0.567907,1.000000,0.451576,0.550233
diff_low,0.570190,0.451576,1.000000,0.541775
diff_close,0.019019,0.550233,0.541775,1.000000


In [8]:
print("Mean absolute |p|:", np.abs(np.corrcoef(stationary_df, rowvar=False).mean()))

Mean absolute |p|: 0.5875873892543888


## VAR (Vector Auto Regression)

In [9]:
from statsmodels.tsa.api import VAR

model = VAR(stationary_df)

c:\Anaconda\envs\var-env\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)


In [10]:
# Select optimal lag using AIC
lag_order = model.select_order(maxlags=30)

print(lag_order.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -41.87      -41.87   6.537e-19      -41.87
1       -45.15      -45.14   2.457e-20      -45.15
2       -45.63      -45.60   1.530e-20      -45.62
3       -45.85      -45.81   1.225e-20      -45.84
4       -45.99      -45.94   1.065e-20      -45.97
5       -46.18      -46.12   8.805e-21      -46.16
6       -46.24      -46.17   8.256e-21      -46.22
7       -46.28      -46.20   7.951e-21      -46.25
8       -46.31      -46.22   7.708e-21      -46.28
9       -46.34      -46.24   7.471e-21      -46.31
10      -46.36      -46.24   7.368e-21      -46.32
11      -46.41      -46.28   6.979e-21      -46.37
12      -46.42      -46.28   6.890e-21      -46.38
13      -46.44      -46.28   6.806e-21      -46.38
14      -46.45      -46.28   6.730e-21      -46.39
15      -46.45      -46.28   6.697e-21      -46.39
16      -46.46      -46.28   6.

In [11]:
print(lag_order.aic)

29


In [12]:
# Fit the model with selected lag
results = model.fit(lag_order.aic)

print(results.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 06, Jun, 2025
Time:                     10:42:51
--------------------------------------------------------------------
No. of Equations:         4.00000    BIC:                   -46.2188
Nobs:                     9970.00    HQIC:                  -46.4425
Log likelihood:           175968.    FPE:                6.03280e-21
AIC:                     -46.5571    Det(Omega_mle):     5.75774e-21
--------------------------------------------------------------------
Results for equation diff_open
                    coefficient       std. error           t-stat            prob
---------------------------------------------------------------------------------
const                 -0.000002         0.000013           -0.115           0.908
L1.diff_open          -0.959329         0.010918          -87.867           0.000
L1.diff_high           0.009878         0.004957    

In [13]:
from statsmodels.stats.stattools import durbin_watson
print(durbin_watson(results.resid))

[2.00297774 2.00197112 1.99923283 2.0005059 ]


## Forecast future values

In [14]:
# Get the last `lag` observations
forecast_input = stationary_df.values[-results.k_ar:]

# Forecast 5 steps ahead
forecast = results.forecast(y=forecast_input, steps=5)

# Convert to DataFrame
forecast_df = pd.DataFrame(forecast, columns=stationary_df.columns)
print(forecast_df)

   diff_open  diff_high  diff_low  diff_close
0  -0.007245  -0.003927 -0.004576   -0.001300
1  -0.001208  -0.001019 -0.001368   -0.000611
2  -0.000666   0.000581 -0.000134   -0.000056
3   0.000106  -0.000553  0.000286   -0.000084
4  -0.000429   0.000045  0.000703    0.001166


In [15]:
def forecast_next(model_res, symbol, timeframe):
    forecast = None
        
    # Get required lags for prediction
    rates = mt5.copy_rates_from_pos(symbol, timeframe, 0, model_res.k_ar+1) # Get rates starting at the current bar to bars=lags used during training
    if rates is None or len(rates) < model_res.k_ar+1:
        print("Failed to get copy rates Error =", mt5.last_error())
        return forecast, None
        
    # Prepare input data and make forecast
    input_data = pd.DataFrame(rates)[["open", "high", "low", "close"]].values
    stationary_input = np.diff(input_data, axis=0)[-model_res.k_ar:] # get the recent values equal to the number of lags used by the model
    
    try:
        forecast = model_res.forecast(stationary_input, steps=1) # predict the next price
    except Exception as e:
        print("Failed to forecast: ", str(e))
        return forecast, None
        
    try:
        updated_data = np.vstack([model_res.endog, stationary_input[-1]]) # concatenate new/last datapoint to the data used during previous training
        updated_model = VAR(updated_data).fit(maxlags=model_res.k_ar) # Retrain the model with new data
    except Exception as e:
        print("Failed to update the model: ", str(e))
        return forecast, None
        
    return forecast, updated_model

In [16]:
res_model = results # Initial model

In [17]:
forecast, res_model = forecast_next(model_res=res_model, symbol=symbol, timeframe=timeframe)

forecast_df = pd.DataFrame(forecast, columns=stationary_df.columns)
print("next forecasted:\n", forecast_df)

next forecasted:
    diff_open  diff_high  diff_low  diff_close
0  -0.001788   0.001743 -0.003324   -0.000326


In [18]:
# mt5.shutdown()